# Fairness check (roadmap step 10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/fairness.ipynb)

How often is human writing wrongly called AI, and is it worse for some writers? This notebook

* downloads the external fairness sets once (`scripts/build_fairness_sets.py`): TOEFL essays by
  non-native writers, US college and grade-8 essays by native writers, Stanford CS224N
  abstracts, AI-edited versions of some of them and ChatGPT-3.5 essays and abstracts
  (Liang et al. 2023, MIT), and 3,911 ELLIPSE essays by English learners in grades 8-12 with
  proficiency scores (CC BY 4.0). Texts that are also in the corpus are dropped;
* scores your saved models on them and on the human texts of the test splits, and reports the
  false-positive rate (human text flagged as AI) per subgroup: by domain, source, length
  (short texts), technical text by length, native vs non-native writers, and English learners by
  proficiency and grade, each with a 95% confidence interval;
* marks groups whose rate is far above the average (at least twice the average, 2 points above
  it, and the confidence interval clear of it) and writes them as known limitations for the app
  (`known_limitations.json`).

**Use a GPU runtime** (T4). Then **Runtime → Run all**. Downloading the sets takes a minute.
Scoring 4,000 to 8,000 texts takes a few minutes per model on a T4 (the transformer and the
GPT-2 features take most of it; features already cached in `CACHE_DIR` are reused), so all
three models take about 15 to 25 minutes.

Writes `My Drive/skolfie/data/processed/fairness/` and `My Drive/skolfie/reports/fairness/`.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
CACHE_DIR = "/content/drive/MyDrive/skolfie/cache"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODELS = ["stacked", "transformer", "features"]  # saved in MODELS_DIR; missing ones skipped
EVAL_SPLITS = "test,test_ood_domain"  # corpus splits whose human texts are measured
MAX_PER_GROUP = 300    # texts per stratum (corpus: domain x length; sets: set x proficiency);
                       # None = all texts (slower, narrower intervals)
MAX_VAL = 5000         # val human texts that set the 1% threshold of models without cut-offs
REBUILD_SETS = False   # True: build the fairness sets again

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU; transformer scoring and GPT-2 features will be very slow.")

from google.colab import drive

drive.mount("/content/drive")
FAIRNESS_DIR = f"{REPORTS_DIR}/fairness"
for d in (FAIRNESS_DIR, f"{DATA_DIR}/fairness", CACHE_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."
assert os.path.exists(f"{DATA_DIR}/splits.parquet"), "Run evaluate.ipynb (step 3) first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Build the fairness sets

Runs only when `fairness_sets.parquet` is missing (or with `REBUILD_SETS = True`). Downloads
are cached in `CACHE_DIR/fairness-downloads`.

In [ ]:
import sys


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


FAIRNESS_SETS = f"{DATA_DIR}/fairness/fairness_sets.parquet"
if REBUILD_SETS or not os.path.exists(FAIRNESS_SETS):
    run(["python", "-u", "scripts/build_fairness_sets.py",
         "--out", FAIRNESS_SETS,
         "--cache-dir", f"{CACHE_DIR}/fairness-downloads",
         "--corpus", f"{DATA_DIR}/corpus.parquet"])
else:
    print("fairness sets already built:", FAIRNESS_SETS)

## 4. Score each model

`stacked` is the product model: its threshold is its own AI cut-off (`stack.ipynb`). The
others have no cut-offs, so theirs is the one that flags 1% of the val human texts. `stacked`
loads its member models from the paths it was saved with, so they must still be in
`MODELS_DIR`.

In [ ]:
present = [m for m in MODELS if os.path.exists(f"{MODELS_DIR}/{m}.joblib")]
print("models:", present, "| missing:", [m for m in MODELS if m not in present])
for m in present:
    cmd = ["python", "-u", "scripts/eval_fairness.py",
           "--corpus", f"{DATA_DIR}/corpus.parquet",
           "--splits", f"{DATA_DIR}/splits.parquet",
           "--fairness-sets", FAIRNESS_SETS,
           "--model", f"{m}={MODELS_DIR}/{m}.joblib",
           "--eval-splits", EVAL_SPLITS,
           "--max-val", str(MAX_VAL),
           "--out-dir", f"{FAIRNESS_DIR}/{m}"]
    if MAX_PER_GROUP:
        cmd += ["--max-per-group", str(MAX_PER_GROUP)]
    run(cmd)

## 5. Read the reports

One report per model. The known limitations of `stacked` (`known_limitations.json`) are what
the app will show.

In [ ]:
from IPython.display import Markdown, display

for m in present:
    with open(f"{FAIRNESS_DIR}/{m}/fairness_report.md", encoding="utf-8") as fh:
        display(Markdown(fh.read()))

Reports are in `My Drive/skolfie/reports/fairness/` (one folder per model). Tell Claude in the
project thread when this has finished; it reads the report from Drive directly.